# Notebook 03 — Pipelines & Evaluation (Kaggle T4)

**Required Inputs:**
1. Kaggle Input dataset: `raddar/chest-xrays-indiana-university`
2. Previous notebook output (with `reports_corpus.csv`, `qa_dataset.jsonl`, `colpali_index/`, `clip_index/`)

**Required Secret:** `HF_TOKEN` (for MedGemma access)

Runs 3 systems on 50 test studies (reduced for time):
- System A: ColPali + MedGemma (RAG)
- System B: CLIP + MedGemma (RAG)
- System C: MedGemma Direct

In [1]:
!pip install -q --upgrade peft transformers
!pip install -q accelerate bitsandbytes colpali-engine open-clip-torch faiss-cpu
!pip install -q bert-score rouge-score
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 105.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 95.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.0 MB/s eta 0:00:00:00:01


In [2]:
import os, sys, subprocess, importlib.util, glob, time
import pandas as pd
from PIL import Image

WORKING_DIR = '/kaggle/working'

# Edit each path to match your Kaggle Input mount points
CORPUS_PATH       = '/kaggle/input/datasets/mohammedtaha778/reports-corpus/reports_corpus.csv'
QA_PATH           = '/kaggle/input/datasets/mohammedtaha778/reports-corpus/qa_dataset.jsonl'
COLPALI_INDEX_DIR = '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/colpali_index'
CLIP_INDEX_DIR    = '/kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/clip_index'

# Verify all exist
for path, name in [(CORPUS_PATH, 'corpus'), (QA_PATH, 'qa_dataset'),
                    (COLPALI_INDEX_DIR, 'colpali_index'), (CLIP_INDEX_DIR, 'clip_index')]:
    exists = '✓' if os.path.exists(path) else '✗'
    print(f'  {exists} {name}: {path}')
    if not os.path.exists(path):
        print(f'    ⚠️  Path does not exist — update the variable above')

  ✓ corpus: /kaggle/input/datasets/mohammedtaha778/reports-corpus/reports_corpus.csv
  ✓ qa_dataset: /kaggle/input/datasets/mohammedtaha778/reports-corpus/qa_dataset.jsonl
  ✓ colpali_index: /kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/colpali_index
  ✓ clip_index: /kaggle/input/notebooks/mohammedtaha778/01-02-data-qa-indexes-complete/clip_index


In [3]:
# Get HF_TOKEN from Kaggle secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
print('✓ HF_TOKEN loaded')

✓ HF_TOKEN loaded


In [4]:
import os, sys, subprocess, importlib.util, glob, time
import pandas as pd
from PIL import Image

REPO_PATH = '/kaggle/working/cxr-rag-system'

# Clone if needed
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mohamedtaha77/cxr-rag-system.git', REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

# CRITICAL: Add repo to sys.path so `from src...` imports work
sys.path.insert(0, REPO_PATH)

# Now use normal imports
from src.generation.medgemma_generator import MedGemmaGenerator
from src.retrieval.colpali_retriever import ColPaliRetriever
from src.retrieval.clip_retriever import CLIPRetriever
from src.evaluation.metrics import Evaluator

print('✓ Modules loaded')

✓ Modules loaded


In [5]:
# Load corpus + test split (limit to 50 for time)
corpus_df = pd.read_csv(CORPUS_PATH)
test_df = corpus_df[corpus_df['split'] == 'test'].head(50).reset_index(drop=True)
study_to_impression = dict(zip(corpus_df['study_id'], corpus_df['impression']))

print(f'Evaluating on {len(test_df)} test studies')

Evaluating on 50 test studies


In [6]:
# Load MedGemma (4-bit, ~3 GB VRAM)
import torch, gc

torch.cuda.empty_cache()
gc.collect()

print('Loading MedGemma (4-bit)...')
generator = MedGemmaGenerator(hf_token=HF_TOKEN, load_in_4bit=True)
print('✓ MedGemma loaded')

Loading MedGemma (4-bit)...


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

✓ MedGemma loaded


In [20]:
from tqdm import tqdm
import time

# Fixed pipeline function
def run_report_pipeline(row, retriever, path_to_impression, k=3):
    image = Image.open(row['image_path']).convert('RGB')
    query = row.get('impression', 'chest x-ray findings')[:100]
    retrieved = retriever.search(query, k=k)
    context = [path_to_impression.get(r.get('image_path', ''), '') for r in retrieved]
    context = [c for c in context if c][:k]
    return generator.generate_report(image, context_reports=context or None)


## System A: ColPali + MedGemma

In [21]:
# System A: ColPali + MedGemma
preds_A, refs_A = [], []
print('\n=== System A: ColPali + MedGemma ===')
start = time.time()
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='System A'):
    pred = run_report_pipeline(row, colpali, path_to_impression)
    preds_A.append(pred)
    refs_A.append(row['impression'])
print(f'Done in {(time.time()-start)/60:.1f} min')

del colpali
gc.collect()
torch.cuda.empty_cache()


=== System A: ColPali + MedGemma ===


System A: 100%|██████████| 50/50 [14:22<00:00, 17.26s/it]


Done in 14.4 min


## System B: CLIP + MedGemma

In [22]:
# System B: CLIP + MedGemma
print('\n=== System B: CLIP + MedGemma ===')
clip = CLIPRetriever()
clip.load_index(CLIP_INDEX_DIR)

class CLIPWrapper:
    def __init__(self, clip):
        self.clip = clip
    def search(self, query, k=3):
        return self.clip.search_by_text(query, k=k)

clip_wrapped = CLIPWrapper(clip)

preds_B, refs_B = [], []
start = time.time()
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='System B'):
    pred = run_report_pipeline(row, clip_wrapped, path_to_impression)
    preds_B.append(pred)
    refs_B.append(row['impression'])
print(f'Done in {(time.time()-start)/60:.1f} min')

del clip, clip_wrapped
gc.collect()
torch.cuda.empty_cache()


=== System B: CLIP + MedGemma ===


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
System B: 100%|██████████| 50/50 [12:29<00:00, 15.00s/it]


Done in 12.5 min


## System C: MedGemma Direct

In [23]:
# System C: MedGemma Direct (no retrieval)
print('\n=== System C: MedGemma Direct ===')
preds_C, refs_C = [], []
start = time.time()
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='System C'):
    image = Image.open(row['image_path']).convert('RGB')
    pred = generator.generate_report(image, context_reports=None)
    preds_C.append(pred)
    refs_C.append(row['impression'])
print(f'Done in {(time.time()-start)/60:.1f} min')

print('\n✓ All 3 systems done. Now run the metrics cell.')


=== System C: MedGemma Direct ===


System C: 100%|██████████| 50/50 [13:15<00:00, 15.91s/it]

Done in 13.3 min

✓ All 3 systems done. Now run the metrics cell.


## Compute Metrics

In [24]:
!pip install -q --upgrade bert_score

In [25]:
# Skip the broken BERTScore, use sentence-transformers instead
!pip install -q sentence-transformers

import types
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

evaluator = Evaluator()

# Replace bertscore with sentence-transformer similarity
sbert = SentenceTransformer('all-MiniLM-L6-v2')

def patched_bertscore(self, predictions, references, model_type=None):
    preds = [str(p)[:2000] for p in predictions]
    refs = [str(r)[:2000] for r in references]
    pred_emb = sbert.encode(preds, show_progress_bar=False)
    ref_emb = sbert.encode(refs, show_progress_bar=False)
    sims = [cosine_similarity([p], [r])[0][0] for p, r in zip(pred_emb, ref_emb)]
    return {
        "BERTScore_P": float(np.mean(sims)),  # Same value for all 3 (single similarity)
        "BERTScore_R": float(np.mean(sims)),
        "BERTScore_F1": float(np.mean(sims)),
    }

evaluator.bertscore = types.MethodType(patched_bertscore, evaluator)

# Run evaluation
results = {}
for label, preds, refs in [
    ('ColPali + MedGemma', preds_A, refs_A),
    ('CLIP + MedGemma', preds_B, refs_B),
    ('MedGemma Direct', preds_C, refs_C),
]:
    print(f'Computing metrics for {label}...')
    results[label] = evaluator.evaluate_report_generation(preds, refs)

results_df = pd.DataFrame(results).T
print('\n=== Results ===')
print(results_df.round(4))

results_df.to_csv(os.path.join(WORKING_DIR, 'results.csv'))

predictions_df = pd.DataFrame({
    'study_id': test_df['study_id'].tolist(),
    'reference': refs_A,
    'colpali_pred': preds_A,
    'clip_pred': preds_B,
    'direct_pred': preds_C,
})
predictions_df.to_csv(os.path.join(WORKING_DIR, 'predictions.csv'), index=False)

print(f'\n✓ Saved results.csv and predictions.csv to {WORKING_DIR}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Computing metrics for ColPali + MedGemma...
[RadGraph skipped] No module named 'radgraph'
Computing metrics for CLIP + MedGemma...
[RadGraph skipped] No module named 'radgraph'
Computing metrics for MedGemma Direct...
[RadGraph skipped] No module named 'radgraph'

=== Results ===
                    BERTScore_P  BERTScore_R  BERTScore_F1  ROUGE-L  \
ColPali + MedGemma       0.4743       0.4743        0.4743   0.0933   
CLIP + MedGemma          0.4590       0.4590        0.4590   0.0898   
MedGemma Direct          0.4614       0.4614        0.4614   0.0750   

                    RadGraph_F1  
ColPali + MedGemma          NaN  
CLIP + MedGemma             NaN  
MedGemma Direct             NaN  

✓ Saved results.csv and predictions.csv to /kaggle/working


## QA Evaluation

In [28]:
from rouge_score import rouge_scorer
import torch, gc

# Reload ColPali if not in memory
if 'colpali' not in dir():
    colpali = ColPaliRetriever.from_index(COLPALI_INDEX_DIR)

# Load QA test set
qa_df = pd.read_json(QA_PATH, lines=True)
qa_test = qa_df[qa_df['split'] == 'test'].head(30).reset_index(drop=True)
print(f'QA evaluation on {len(qa_test)} pairs')

# Run QA inference
qa_preds, qa_refs = [], []
start = time.time()
for _, row in tqdm(qa_test.iterrows(), total=len(qa_test), desc='QA'):
    image = Image.open(row['image_path']).convert('RGB')
    retrieved = colpali.search(row['question'], k=3)
    # Use path-based lookup (fixed)
    context = [path_to_impression.get(r.get('image_path', ''), '') for r in retrieved]
    context = [c for c in context if c][:3]
    pred = generator.answer_question(image, row['question'], context)
    qa_preds.append(pred)
    qa_refs.append(row['answer'])

elapsed_qa = time.time() - start
print(f'✓ QA inference done in {elapsed_qa/60:.1f} min')

# Compute metrics (skip broken BLEU)
valid = [(p, r) for p, r in zip(qa_preds, qa_refs) if p and r]
print(f'Valid pairs: {len(valid)}/{len(qa_preds)}')

if valid:
    valid_preds, valid_refs = zip(*valid)
    bert_metrics = evaluator.bertscore(list(valid_preds), list(valid_refs))
    
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(r, p)['rougeL'].fmeasure for p, r in zip(valid_preds, valid_refs)]
    rouge_l = sum(rouge_scores) / len(rouge_scores)
    
    qa_metrics = {**bert_metrics, 'ROUGE-L': rouge_l}
    print(f'\nQA Metrics: {qa_metrics}')

# Save
qa_results_df = pd.DataFrame({
    'question': qa_test['question'].tolist(),
    'reference': qa_refs,
    'prediction': qa_preds,
})
qa_results_df.to_csv(os.path.join(WORKING_DIR, 'qa_results.csv'), index=False)
print(f'\n✓ Saved qa_results.csv to {WORKING_DIR}')

QA evaluation on 0 pairs


QA: 0it [00:00, ?it/s]

✓ QA inference done in 0.0 min
Valid pairs: 0/0

✓ Saved qa_results.csv to /kaggle/working


In [29]:
# Check QA dataset structure
qa_df = pd.read_json(QA_PATH, lines=True)
print(f'Total QA pairs: {len(qa_df)}')
print(f'\nSplit distribution:')
print(qa_df['split'].value_counts())
print(f'\nFirst 3 rows:')
print(qa_df.head(3))

Total QA pairs: 1515

Split distribution:
split
train    1515
Name: count, dtype: int64

First 3 rows:
   study_id                                         image_path       category  \
0      1432  /kaggle/input/datasets/raddar/chest-xrays-indi...  Consolidation   
1      1432  /kaggle/input/datasets/raddar/chest-xrays-indi...  Consolidation   
2      1432  /kaggle/input/datasets/raddar/chest-xrays-indi...  Consolidation   

                                            question  \
0  Is there consolidation visible in this chest X...   
1         Can pulmonary consolidation be identified?   
2         Are there signs of airspace consolidation?   

                                              answer  split  
0         No consolidation is observed in the lungs.  train  
1  No, pulmonary consolidation is not observed. T...  train  
2  No, airspace consolidation is not observed. Th...  train  


In [30]:
from rouge_score import rouge_scorer
import torch, gc

# Reload ColPali if needed
if 'colpali' not in dir():
    colpali = ColPaliRetriever.from_index(COLPALI_INDEX_DIR)

# Use first 30 pairs (no test split available)
qa_df = pd.read_json(QA_PATH, lines=True)
qa_test = qa_df.head(30).reset_index(drop=True)
print(f'QA evaluation on {len(qa_test)} pairs (from train split — no test split available)')

# Run QA inference
qa_preds, qa_refs = [], []
start = time.time()
for _, row in tqdm(qa_test.iterrows(), total=len(qa_test), desc='QA'):
    image = Image.open(row['image_path']).convert('RGB')
    retrieved = colpali.search(row['question'], k=3)
    context = [path_to_impression.get(r.get('image_path', ''), '') for r in retrieved]
    context = [c for c in context if c][:3]
    pred = generator.answer_question(image, row['question'], context)
    qa_preds.append(pred)
    qa_refs.append(row['answer'])

elapsed_qa = time.time() - start
print(f'✓ QA inference done in {elapsed_qa/60:.1f} min')

# Compute metrics
valid = [(p, r) for p, r in zip(qa_preds, qa_refs) if p and r]
print(f'Valid pairs: {len(valid)}/{len(qa_preds)}')

if valid:
    valid_preds, valid_refs = zip(*valid)
    bert_metrics = evaluator.bertscore(list(valid_preds), list(valid_refs))
    
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(r, p)['rougeL'].fmeasure for p, r in zip(valid_preds, valid_refs)]
    rouge_l = sum(rouge_scores) / len(rouge_scores)
    
    qa_metrics = {**bert_metrics, 'ROUGE-L': rouge_l}
    print(f'\nQA Metrics: {qa_metrics}')

# Save
qa_results_df = pd.DataFrame({
    'question': qa_test['question'].tolist(),
    'reference': qa_refs,
    'prediction': qa_preds,
    'category': qa_test['category'].tolist(),
})
qa_results_df.to_csv(os.path.join(WORKING_DIR, 'qa_results.csv'), index=False)
print(f'\n✓ Saved qa_results.csv to {WORKING_DIR}')

QA evaluation on 30 pairs (from train split — no test split available)


QA: 100%|██████████| 30/30 [06:19<00:00, 12.65s/it]

✓ QA inference done in 6.3 min
Valid pairs: 30/30

QA Metrics: {'BERTScore_P': 0.6696304678916931, 'BERTScore_R': 0.6696304678916931, 'BERTScore_F1': 0.6696304678916931, 'ROUGE-L': 0.2040059769391999}

✓ Saved qa_results.csv to /kaggle/working
